# 05 - Prédire NOVA et le Nutri-Score : modèles de référence

**Objectif 3** : que peut-on deviner du degré de transformation et de la qualité nutritionnelle d'un produit à partir de sa description ?

**Quatre tâches, deux scénarios par cible**

| Tâche | Entrée | Rôle |
|---|---|---|
| `nova_name` | nom + catégories | **Scénario A** : la vraie question. Utile pour les ~73 % de produits sans liste d'ingrédients |
| `nova_full` | + liste d'ingrédients | **Scénario B** : contrôle. NOVA est calculé par règles à partir des ingrédients |
| `nutriscore_text` | nom + catégories + ingrédients | **Scénario A** : le texte suffit-il à deviner la qualité nutritionnelle ? |
| `nutriscore_full` | + valeurs nutritionnelles | **Scénario B** : contrôle. Le Nutri-Score est calculé à partir des nutriments |

Dans chaque paire, les deux scénarios portent sur **les mêmes produits et le même découpage** (70 % entraînement, 15 % validation, 15 % test, stratifié). L'écart entre A et B mesure donc ce qu'apportent les informations ajoutées.

**Deux modèles de référence** : TF-IDF (mots et paires de mots, 50 000 termes) + régression logistique, et TF-IDF + XGBoost (5 000 termes les plus liés à la cible, sélectionnés par test du chi²).

**Choix fondé sur un essai** (50 000 produits par tâche) : sur le texte seul, la régression logistique fait aussi bien que XGBoost, 20 à 30 fois plus vite ; quand il faut retrouver une règle de calcul (scénarios B), XGBoost gagne nettement (+8 points de F1 sur le Nutri-Score), car les règles fonctionnent par seuils, comme les arbres. XGBoost n'est donc entraîné que pour les scénarios B sur la base complète. Le transformer (CamemBERT) viendra à l'étape 7, sur le même jeu de test.

**Métriques** (calculées sur le jeu de test uniquement)
- **F1 macro**, métrique principale : chaque classe pèse autant, même les plus rares.
- **Accuracy** : part de prédictions exactes.
- **À une classe près** : NOVA et Nutri-Score sont ordonnés ; confondre B et C est moins grave que A et E.

Prérequis :
```
uv run python -m nutrilys.models.baseline --models lr
uv run python -m nutrilys.models.baseline --tasks nova_full nutriscore_full --models xgb
```

In [21]:
import json

import pandas as pd

from nutrilys import config
from nutrilys.db import products_connection
from nutrilys.models import baseline
from nutrilys.models.dataset import TASKS
from nutrilys.models.evaluate import confusion_shares, per_class_f1
from nutrilys.viz import charts
from nutrilys.viz.fmt import fmt_int, fmt_num, fmt_pct
from nutrilys.viz.theme import NOVA_LABELS, SERIES, save_figure, use_theme

use_theme()
con = products_connection()
metrics = pd.read_csv(config.MODEL_REPORTS_DIR / "baseline_metrics.csv")
run_info = json.loads((config.MODEL_REPORTS_DIR / "baseline_run.json").read_text(encoding="utf-8"))
findings = []
if run_info.get("sample"):
    print(
        f"Attention : modèles entraînés sur un échantillon de {fmt_int(run_info['sample'])} "
        "produits par tâche (essai rapide)."
    )

## 1. Vue d'ensemble des résultats

In [22]:
order = list(TASKS)
metrics["task_label"] = metrics["task"].map({k: t.description for k, t in TASKS.items()})
metrics = (
    metrics.set_index("task").loc[[t for t in order if t in set(metrics["task"])]].reset_index()
)

view = metrics[
    [
        "task_label",
        "model",
        "f1_macro",
        "accuracy",
        "within_one",
        "n_train",
        "fit_seconds",
        "predict_ms_per_1000",
    ]
].copy()
view["model"] = view["model"].map(baseline.MODELS)
view.columns = [
    "Tâche",
    "Modèle",
    "F1 macro",
    "Accuracy",
    "À une classe près",
    "Produits d'entraînement",
    "Entraînement (s)",
    "Prédiction (ms / 1000)",
]
view.style.format(
    {
        "F1 macro": "{:.3f}",
        "Accuracy": "{:.3f}",
        "À une classe près": "{:.3f}",
        "Produits d'entraînement": "{:,.0f}",
        "Entraînement (s)": "{:.0f}",
        "Prédiction (ms / 1000)": "{:.0f}",
    }
).hide(axis="index")

Tâche,Modèle,F1 macro,Accuracy,À une classe près,Produits d'entraînement,Entraînement (s),Prédiction (ms / 1000)
NOVA à partir du nom et des catégories,Régression logistique,0.833,0.825,0.949,"223,666",67,70
NOVA avec la liste d'ingrédients,Régression logistique,0.951,0.956,0.984,"223,666",118,163
NOVA avec la liste d'ingrédients,XGBoost,0.957,0.961,0.985,"223,666",1414,291
Nutri-Score à partir du texte,Régression logistique,0.721,0.735,0.942,"190,549",191,200
Nutri-Score avec les valeurs nutritionnelles,Régression logistique,0.821,0.835,0.985,"190,549",227,164
Nutri-Score avec les valeurs nutritionnelles,XGBoost,0.895,0.907,0.991,"190,549",1401,206


In [23]:
best = metrics.loc[metrics.groupby("task")["f1_macro"].idxmax()].set_index("task")["f1_macro"]
fig = charts.grouped_bars(
    metrics,
    "task_label",
    "model",
    "f1_macro",
    {"lr": SERIES[0], "xgb": SERIES[1]},
    series_labels=baseline.MODELS,
    title=f"NOVA depuis le nom seul : F1 macro {fmt_num(best.get('nova_name', 0), 2)}, "
    f"{fmt_num(best.get('nova_full', 0), 2)} avec les ingrédients",
    subtitle="F1 macro sur le jeu de test, par tâche et par modèle",
    value_label="F1 macro",
    value_range=(0, 1.08),
)
save_figure(fig, "baseline_f1")
fig.show()

In [24]:
for target, (a, b) in {
    "NOVA": ("nova_name", "nova_full"),
    "Nutri-Score": ("nutriscore_text", "nutriscore_full"),
}.items():
    if a in best and b in best:
        findings.append(
            f"{target} : F1 macro {fmt_num(best[a], 2)} en scénario A "
            f"({TASKS[a].description}), {fmt_num(best[b], 2)} en scénario B."
        )
gap = metrics.pivot(index="task", columns="model", values="f1_macro")
if {"lr", "xgb"} <= set(gap.columns):
    diff = (gap["xgb"] - gap["lr"]).abs().max()
    findings.append(
        f"Régression logistique et XGBoost : écart de F1 macro d'au plus "
        f"{fmt_num(diff, 3)} selon la tâche."
    )

**Comment lire les scénarios B.** Un score proche de 1 n'est pas une prouesse : NOVA et le Nutri-Score sont *calculés* par Open Food Facts à partir des ingrédients et des nutriments, et le modèle retrouve ces règles. Ces scénarios servent de plafond de référence et de contrôle de cohérence. Le Nutri-Score ne peut pas atteindre exactement 1, car sa formule utilise aussi la part de fruits, légumes et légumineuses, que nous ne donnons pas au modèle.

**Le vrai résultat est l'écart entre A et B** : il mesure ce que le seul texte permet de deviner.

## 2. NOVA à partir du nom : où le modèle se trompe-t-il ?

In [25]:
model_a = baseline.best_model(metrics, "nova_name")
pred_a = pd.read_parquet(config.PREDICTIONS_DIR / f"nova_name_{model_a}.parquet")
labels = [str(c) for c in config.NOVA_GROUPS]
shares = confusion_shares(pred_a["y_true"] - 1, pred_a["y_pred"] - 1, labels)
recall4 = shares.loc["4", "4"]

fig = charts.confusion_heatmap(
    shares,
    title=f"{fmt_pct(recall4)} des produits ultra-transformés sont reconnus depuis leur nom",
    subtitle=f"Matrice de confusion, NOVA depuis le nom et les catégories "
    f"({baseline.MODELS[model_a]}), % de chaque groupe réel",
    x_title="Groupe NOVA prédit",
    y_title="Groupe NOVA réel",
)
save_figure(fig, "nova_name_confusion")
findings.append(f"NOVA depuis le nom : {fmt_pct(recall4)} des NOVA 4 correctement reconnus.")
fig.show()

In [26]:
per_class_f1(
    pred_a["y_true"] - 1, pred_a["y_pred"] - 1, [NOVA_LABELS[g] for g in config.NOVA_GROUPS]
).to_frame("F1").style.format("{:.3f}")

,F1
1 - Brut ou peu transformé,0.842
2 - Ingrédient culinaire,0.919
3 - Transformé,0.706
4 - Ultra-transformé,0.867


**Lecture attendue** : le groupe 2 (ingrédients culinaires : huile, beurre, sucre, sel) et le groupe 1 se reconnaissent bien à leur nom. Le groupe 3 (transformés) est le plus difficile : un « pain » ou un « jambon » peut être classé 3 ou 4 selon la présence d'un seul additif ou d'un arôme, invisible dans le nom.

## 3. Ce que le modèle a appris

Termes les plus associés à chaque groupe NOVA par la régression logistique (poids les plus élevés). C'est ce qui rend ce modèle de référence explicable.

In [27]:
lr_name = baseline.load_model("nova_name", "lr")
terms = baseline.top_terms(lr_name, n=12)
table = pd.DataFrame(
    {NOVA_LABELS[g]: terms.loc[terms["class"] == g, "term"].to_list() for g in config.NOVA_GROUPS}
)
table

,1 - Brut ou peu transformé,2 - Ingrédient culinaire,3 - Transformé,4 - Ultra-transformé
0,pur jus,fats,meals,saveur
1,complet plant,salts,cheeses,aromatisee
2,processed frozen,vinegars,beverages alcoholic,aromatise
3,moulu,sugars,products cheeses,stevia
4,products potatoes,starches,wines,aromatises
5,dumplings pastas,honeys,cheeses dairy,caramel
6,sans sel,oils,nectar,whey
7,vpf,sweeteners honeys,dessert de,gout
8,puree pomme,maple syrups,snacks,potatoes meals
9,crue,maple,terrines,sweet snacks


## 4. Exemples d'erreurs franches

Produits ultra-transformés que le modèle, au vu du seul nom, croyait bruts ou peu transformés, avec une forte confiance. Ces cas montrent les limites du texte seul, et peuvent signaler des produits dont le nom « rassure » à tort.

In [28]:
errors = (
    pred_a[(pred_a["y_true"] == 4) & (pred_a["y_pred"] == 1)]
    .sort_values("confidence", ascending=False)
    .head(15)
)
con.register("errors", errors)
examples = con.execute("""
    SELECT p.product_name AS "Produit", p.brand AS "Marque",
           round(100 * e.confidence) AS "Confiance (%)",
           left(p.ingredients_text, 120) AS "Début de la liste d'ingrédients"
    FROM errors e JOIN products p USING (code)
    ORDER BY e.confidence DESC
""").df()
examples

,Produit,Marque,Confiance (%),Début de la liste d'ingrédients
0,Jus d’ananas pressé,Andros,99.0,"Jus d'ananas, stabilisant : pectine de fruits."
1,"Riz complet, Orge & Quinoa",Graines de Nature Seeds of change,99.0,"riz complet&quot; précuit à la vapeur (66%), o..."
2,Fusilli,FiberPasta,99.0,"Semoule de blé dur, fibre de blé (12%), inulin..."
3,Mâche,Excellent Buffet,99.0,"Sweet potatoes (sweet potato, canola oil, hone..."
4,Eau water better,Better,99.0,"Eau, arôme naturel de pomme, arôme naturel de ..."
5,Farine de blé ordinaire,Mother's Choice,99.0,"farine de blé, farine de malt de blé, gluten d..."
6,Saskia mineralwasser,Saguaro,99.0,"natural mineral water, carbon dioxide"
7,Café Chicco d‘Oro Tradition - Bohnen,6828 Balerna,99.0,CAFEE CHICCO DORO TRADITION AD ALTISSIMO CONTE...
8,WHOLE BLACK PEPPERCORNS,NaN,99.0,100% whole black peppercons. a factory which u...
9,Aqua minerale naturale,NaN,99.0,"Temperatura alla sorgente: 25,6 °C; pH alla te..."


## 5. Application : estimer NOVA pour les produits sans liste d'ingrédients

Le modèle « nom et catégories » est appliqué aux produits qui n'ont pas de groupe NOVA. Ces estimations alimenteront l'application, clairement présentées comme des **estimations**.

In [29]:
estimated = pd.read_parquet(config.NOVA_ESTIMATED_PATH)
known = con.execute(
    "SELECT nova_group AS predicted, count(*) AS n FROM products WHERE has_nova GROUP BY 1"
).df()
dist = (
    estimated["predicted"]
    .value_counts(normalize=True)
    .mul(100)
    .rename("Estimé (sans ingrédients)")
    .to_frame()
    .join(
        known.set_index("predicted")["n"]
        .pipe(lambda s: 100 * s / s.sum())
        .rename("Connu (avec ingrédients)")
    )
    .sort_index()
)
dist.index = [NOVA_LABELS[g] for g in dist.index]
share4 = dist.iloc[3, 0]
confident = (estimated["confidence"] >= 0.7).mean() * 100

long = dist.reset_index(names="NOVA").melt(id_vars="NOVA", var_name="source", value_name="share")
fig = charts.grouped_bars(
    long,
    "NOVA",
    "source",
    "share",
    {"Connu (avec ingrédients)": SERIES[0], "Estimé (sans ingrédients)": SERIES[2]},
    title=f"{fmt_pct(share4)} des produits sans liste d'ingrédients seraient ultra-transformés",
    subtitle=f"{fmt_int(len(estimated))} produits estimés depuis leur nom ; "
    f"{fmt_pct(confident)} avec une confiance d'au moins 70 %",
    value_label="Part des produits (%)",
    value_format=".0f",
)
save_figure(fig, "nova_estimated")
findings.append(
    f"Estimation : {fmt_int(len(estimated))} produits sans ingrédients, dont "
    f"{fmt_pct(share4)} estimés ultra-transformés ({fmt_pct(confident)} avec une "
    "confiance d'au moins 70 %)."
)
fig.show()

**Prudence** : ces produits sont moins bien documentés que ceux qui ont servi à l'entraînement (pas de liste d'ingrédients, souvent pas de catégorie). Le modèle y est probablement moins fiable que sur le jeu de test. Il faut lire cette répartition comme un ordre de grandeur, pas comme une mesure.

## 6. Synthèse

In [30]:
for line in findings:
    print("-", line)

- NOVA : F1 macro 0,83 en scénario A (NOVA à partir du nom et des catégories), 0,96 en scénario B.
- Nutri-Score : F1 macro 0,72 en scénario A (Nutri-Score à partir du texte), 0,89 en scénario B.
- Régression logistique et XGBoost : écart de F1 macro d'au plus 0,074 selon la tâche.
- NOVA depuis le nom : 82 % des NOVA 4 correctement reconnus.
- Estimation : 831 123 produits sans ingrédients, dont 69 % estimés ultra-transformés (74 % avec une confiance d'au moins 70 %).


## 7. Limites

- **Cibles calculées par des règles** : NOVA et le Nutri-Score ne sont pas des vérités terrain mesurées, mais le résultat des règles d'Open Food Facts. Le modèle apprend à reproduire ces règles, avec leurs propres limites (un seul arôme suffit à classer en NOVA 4).
- **Décalage de population** : les produits sans ingrédients, auxquels on applique le modèle « nom », ne ressemblent pas tout à fait aux produits d'entraînement.
- **Texte multilingue** : quelques fiches sont en anglais ou dans une autre langue. TF-IDF les traite comme un vocabulaire à part.
- **Modèles de référence** : volontairement simples. L'étape 7 dira si un modèle de langage (CamemBERT) fait mieux sur le même jeu de test, et à quel coût.